# 02 - Baseline Transformer + MIA (point de depart propre)

Objectif:
- Construire une baseline unique avant defenses.
- Evaluer l'attaque MIA en mode black-box sur donnees potentiellement biaisees/fuyantes.

Regles de ce notebook:
- Un seul pipeline baseline (pas de legacy single-run).
- Protocole unifie multi-seeds pour comparaison avec 03-06.
- Protocole robuste black-box (meta + LiRA-like + boundary) pour cas reel.

In [1]:
import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import accuracy_score, roc_auc_score
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping

from research_protocol import evaluate_mia_research_protocol, set_seed
from research_protocol_robust import evaluate_mia_robust_protocol

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)


def set_global_determinism(seed: int) -> None:
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass

In [2]:
SEED = 42
ATTACK_SEEDS = [11, 22, 33, 44, 55]

# Baseline de securite: on teste un cas plus vulnerable (fuite/overfit possible)
TARGET_DROPOUT_SAFE = 0.15
TARGET_DROPOUT_RISKY = 0.00
TARGET_EPOCHS_SAFE = 120
TARGET_EPOCHS_RISKY = 260

# Attaquant plus fort pour avoir une baseline MIA exploitable
N_SHADOWS = 30
SHADOW_EPOCHS = 80
SHADOW_BATCH = 16

ROBUST_N_SHADOWS = 40
ROBUST_SHADOW_EPOCHS = 80
ROBUST_BOUNDARY_DIRS = 32
ROBUST_BOUNDARY_STEPS = 10
ROBUST_BOUNDARY_MAX_ALPHA = 2.0

ATTACK_MIN_AUC_TARGET = 0.55

set_global_determinism(SEED)

prepared_path = os.path.join('..', 'data_preparation', 'prepared_oasis2_transformer.npz')
if not os.path.exists(prepared_path):
    prepared_path = os.path.join('data_preparation', 'prepared_oasis2_transformer.npz')

if not os.path.exists(prepared_path):
    raise FileNotFoundError('prepared_oasis2_transformer.npz not found')

b = np.load(prepared_path)

X = b['X'].astype(np.float32)
y = b['y'].astype(np.int32)
X_train = b['X_train'].astype(np.float32)
X_test = b['X_test'].astype(np.float32)
y_train = b['y_train'].astype(np.int32)
y_test = b['y_test'].astype(np.int32)

if 'X_shadow' in b:
    X_shadow_raw = b['X_shadow'].astype(np.float32)
else:
    X_shadow_raw = b['X_shadow_raw'].astype(np.float32)

y_shadow = b['y_shadow'].astype(np.int32)

X_train_seq = X_train.reshape(-1, 1, X_train.shape[1])
X_test_seq = X_test.reshape(-1, 1, X_test.shape[1])

print(f'Loaded: X_train={X_train_seq.shape}, X_test={X_test_seq.shape}, X_shadow={X_shadow_raw.shape}')
print(f'Class ratio train={y_train.mean():.4f}, test={y_test.mean():.4f}')


def build_transformer(n_features, dropout=0.15):
    inp = layers.Input(shape=(1, n_features))
    x = layers.Dense(64)(inp)
    a = layers.MultiHeadAttention(num_heads=4, key_dim=16, dropout=dropout)(x, x)
    x = layers.LayerNormalization(epsilon=1e-6)(x + a)
    f = layers.Dense(128, activation='relu')(x)
    f = layers.Dropout(dropout)(f)
    f = layers.Dense(64)(f)
    x = layers.LayerNormalization(epsilon=1e-6)(x + f)
    x = layers.Flatten()(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(dropout)(x)
    out = layers.Dense(1, activation='sigmoid')(x)

    model = Model(inp, out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy'],
    )
    return model

Loaded: X_train=(70, 1, 9), X_test=(284, 1, 9), X_shadow=(107, 9)
Class ratio train=0.3571, test=0.3592


In [3]:
print('=== Baseline utility model (safe) ===')
set_seed(SEED + 1)
tf.keras.backend.clear_session()

model_safe = build_transformer(n_features=X_train.shape[1], dropout=TARGET_DROPOUT_SAFE)
es = EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True, verbose=0)
model_safe.fit(
    X_train_seq,
    y_train,
    epochs=TARGET_EPOCHS_SAFE,
    batch_size=32,
    validation_split=0.2,
    callbacks=[es],
    verbose=0,
)

p_tr_safe = model_safe.predict(X_train_seq, verbose=0).ravel()
p_te_safe = model_safe.predict(X_test_seq, verbose=0).ravel()

print(pd.DataFrame([{
    'model': 'safe_reference',
    'train_acc': accuracy_score(y_train, (p_tr_safe >= 0.5).astype(int)),
    'test_acc': accuracy_score(y_test, (p_te_safe >= 0.5).astype(int)),
    'gap': accuracy_score(y_train, (p_tr_safe >= 0.5).astype(int)) - accuracy_score(y_test, (p_te_safe >= 0.5).astype(int)),
    'train_auc': roc_auc_score(y_train, p_tr_safe),
    'test_auc': roc_auc_score(y_test, p_te_safe),
}]).round(4))

print('\n=== Baseline attack target (risky / fuite-potentielle) ===')
set_seed(SEED + 2)
tf.keras.backend.clear_session()

model_attack = build_transformer(n_features=X_train.shape[1], dropout=TARGET_DROPOUT_RISKY)
model_attack.fit(
    X_train_seq,
    y_train,
    epochs=TARGET_EPOCHS_RISKY,
    batch_size=32,
    verbose=0,
)

p_tr_attack = model_attack.predict(X_train_seq, verbose=0).ravel()
p_te_attack = model_attack.predict(X_test_seq, verbose=0).ravel()

print(pd.DataFrame([{
    'model': 'attack_target',
    'train_acc': accuracy_score(y_train, (p_tr_attack >= 0.5).astype(int)),
    'test_acc': accuracy_score(y_test, (p_te_attack >= 0.5).astype(int)),
    'gap': accuracy_score(y_train, (p_tr_attack >= 0.5).astype(int)) - accuracy_score(y_test, (p_te_attack >= 0.5).astype(int)),
    'train_auc': roc_auc_score(y_train, p_tr_attack),
    'test_auc': roc_auc_score(y_test, p_te_attack),
}]).round(4))

print('\n=== MIA standard (attaquant fort) ===')
baseline_summary, baseline_per_seed = evaluate_mia_research_protocol(
    target_model=model_attack,
    X_train_seq=X_train_seq,
    y_train=y_train,
    X_test_seq=X_test_seq,
    y_test=y_test,
    X_shadow_raw=X_shadow_raw,
    y_shadow=y_shadow,
    model_builder=lambda nf: build_transformer(n_features=nf, dropout=TARGET_DROPOUT_RISKY),
    seed_list=ATTACK_SEEDS,
    n_shadows=N_SHADOWS,
    shadow_epochs=SHADOW_EPOCHS,
    shadow_batch_size=SHADOW_BATCH,
)
display(baseline_summary.round(4))

print('\n=== MIA robuste (black-box fort) ===')
baseline_robust_summary, baseline_robust_per_seed = evaluate_mia_robust_protocol(
    target_model=model_attack,
    x_train_seq=X_train_seq,
    y_train=y_train,
    x_test_seq=X_test_seq,
    y_test=y_test,
    x_shadow_raw=X_shadow_raw,
    y_shadow=y_shadow,
    model_builder=lambda nf: build_transformer(n_features=nf, dropout=TARGET_DROPOUT_RISKY),
    seed_list=ATTACK_SEEDS,
    n_shadows=ROBUST_N_SHADOWS,
    shadow_epochs=ROBUST_SHADOW_EPOCHS,
    shadow_batch_size=SHADOW_BATCH,
    boundary_dirs=ROBUST_BOUNDARY_DIRS,
    boundary_steps=ROBUST_BOUNDARY_STEPS,
    boundary_max_alpha=ROBUST_BOUNDARY_MAX_ALPHA,
)
display(baseline_robust_summary.round(4))

print('\nQuick view (AUC):')
quick_standard = baseline_summary[['attack', 'auc_mean', 'auc_std']].sort_values('auc_mean', ascending=False)
display(quick_standard.round(4))
quick_robust = baseline_robust_per_seed[['seed', 'auc', 'selected_scorer', 'tpr_at_1pct_fpr', 'tpr_at_5pct_fpr']]
display(quick_robust.round(4))

std_auc = float(quick_standard.iloc[0]['auc_mean'])
rob_auc = float(baseline_robust_summary.iloc[0]['auc_mean'])
if std_auc < ATTACK_MIN_AUC_TARGET and rob_auc < ATTACK_MIN_AUC_TARGET:
    print(f"\nWARNING: attaques encore faibles (std={std_auc:.4f}, robust={rob_auc:.4f}).")
    print('Pour un benchmark defense agressif, augmenter encore N_SHADOWS/epochs ou reduire train size.')

=== Baseline utility model (safe) ===



c:\Users\khalil\AppData\Local\Programs\Python\Python39\lib\site-packages\keras\src\ops\nn.py:944: UserWarning: You are using a softmax over axis 3 of a tensor of shape (None, 4, 1, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(
c:\Users\khalil\AppData\Local\Programs\Python\Python39\lib\site-packages\keras\src\ops\nn.py:944: UserWarning: You are using a softmax over axis 3 of a tensor of shape (32, 4, 1, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(


            model  train_acc  test_acc     gap  train_auc  test_auc
0  safe_reference     0.9571    0.9155  0.0416     0.9902    0.9849

=== Baseline attack target (risky / fuite-potentielle) ===
           model  train_acc  test_acc     gap  train_auc  test_auc
0  attack_target        1.0    0.9296  0.0704        1.0    0.9904

=== MIA standard (attaquant fort) ===


,attack,auc_mean,auc_std,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std
1,shadow_meta,0.4976,0.0323,0.3465,0.0270,0.2160,0.0069,0.8786,0.0196,0.3466,0.0092
0,logistic,0.4608,0.0166,0.8028,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
2,threshold_loss,0.4608,0.0166,0.2690,0.0135,0.2125,0.0031,1.0000,0.0000,0.3505,0.0042



=== MIA robuste (black-box fort) ===


,attack,auc_mean,auc_std,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std
0,shadow_meta,0.6285,0.0535,0.5141,0.0282,0.2385,0.0244,0.6714,0.1053,0.3515,0.0396



Quick view (AUC):


,attack,auc_mean,auc_std
1,shadow_meta,0.4976,0.0323
0,logistic,0.4608,0.0166
2,threshold_loss,0.4608,0.0166


,seed,auc,selected_scorer,tpr_at_1pct_fpr,tpr_at_5pct_fpr
0,11,0.6858,lira_only,0.0714,0.2143
1,22,0.6031,meta_only,0.0000,0.0357
2,33,0.6662,fusion_0.30_0.30_0.40,0.0000,0.1786
3,44,0.6369,fusion_0.30_0.30_0.40,0.0357,0.0714
4,55,0.5508,fusion_0.30_0.30_0.40,0.0357,0.0357


## Detection de patterns d'attaque MIA via logs (simulation cas reel)

Idee: au lieu de seulement mesurer la performance d'une attaque MIA, on observe les **traces de requetes** vers le modele cible.
On simule deux types de sessions:
- **Benignes**: requetes variees, peu de repetition, distribution de confiance plus stable.
- **Attaquantes**: requetes repetees/proche-frontiere (probabilites proches de 0.5), perturbations successives d'un meme point.

Ensuite on construit un detecteur de session suspecte pour estimer si une attaque MIA est en cours.

In [4]:
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(SEED + 999)

def prob_entropy(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return -(p * np.log(p) + (1 - p) * np.log(1 - p))

def build_session_features(probs, points_flat, session_id, label):
    probs = np.asarray(probs, dtype=np.float32)
    points_flat = np.asarray(points_flat, dtype=np.float32)

    rounded = np.round(points_flat, 3)
    _, uniq_idx = np.unique(rounded, axis=0, return_index=True)
    unique_ratio = float(len(uniq_idx) / max(1, len(points_flat)))

    ent = prob_entropy(probs)
    margin = np.abs(probs - 0.5)
    near_boundary_ratio = float(np.mean((probs > 0.45) & (probs < 0.55)))

    return {
        'session_id': session_id,
        'label_attack': int(label),
        'n_queries': int(len(probs)),
        'unique_ratio': unique_ratio,
        'repeat_ratio': 1.0 - unique_ratio,
        'p_mean': float(np.mean(probs)),
        'p_std': float(np.std(probs)),
        'max_conf_mean': float(np.mean(np.maximum(probs, 1 - probs))),
        'max_conf_std': float(np.std(np.maximum(probs, 1 - probs))),
        'entropy_mean': float(np.mean(ent)),
        'entropy_std': float(np.std(ent)),
        'margin_mean': float(np.mean(margin)),
        'margin_std': float(np.std(margin)),
        'near_boundary_ratio': near_boundary_ratio,
    }

# 1) Construire des sessions benignes
n_benign_sessions = 240
queries_per_benign = 30

X_test_flat = X_test_seq.reshape(X_test_seq.shape[0], -1)
benign_rows = []

for s in range(n_benign_sessions):
    idx = rng.choice(X_test_flat.shape[0], size=queries_per_benign, replace=False)
    pts = X_test_flat[idx]
    pts_seq = pts.reshape(-1, 1, X_train.shape[1])
    probs = model_attack.predict(pts_seq, verbose=0).ravel()
    benign_rows.append(build_session_features(probs, pts, f'benign_{s}', 0))

# 2) Construire des sessions attaquantes (requetes repetitives + petites perturbations)
n_attack_sessions = 240
anchors_per_session = 5
queries_per_anchor = 8
noise_std = 0.03

attack_rows = []

for s in range(n_attack_sessions):
    anchor_idx = rng.choice(X_test_flat.shape[0], size=anchors_per_session, replace=False)
    anchors = X_test_flat[anchor_idx]

    session_points = []
    for a in anchors:
        for _ in range(queries_per_anchor):
            noise = rng.normal(0.0, noise_std, size=a.shape).astype(np.float32)
            session_points.append(a + noise)

    pts = np.asarray(session_points, dtype=np.float32)
    pts_seq = pts.reshape(-1, 1, X_train.shape[1])
    probs = model_attack.predict(pts_seq, verbose=0).ravel()
    attack_rows.append(build_session_features(probs, pts, f'attack_{s}', 1))

df_logs = pd.DataFrame(benign_rows + attack_rows)
display(df_logs.head())
print('Sessions total:', len(df_logs), '| Benignes:', (df_logs['label_attack'] == 0).sum(), '| Attaquantes:', (df_logs['label_attack'] == 1).sum())

# 3) Detection supervisee (si historique annote disponible)
feature_cols = [c for c in df_logs.columns if c not in ['session_id', 'label_attack']]
X_feat = df_logs[feature_cols].astype(np.float32).values
y_feat = df_logs['label_attack'].values

Xtr, Xva, ytr, yva = train_test_split(
    X_feat, y_feat, test_size=0.30, random_state=SEED, stratify=y_feat
)

clf = RandomForestClassifier(n_estimators=300, random_state=SEED, class_weight='balanced')
clf.fit(Xtr, ytr)
p_attack = clf.predict_proba(Xva)[:, 1]
auc_det = roc_auc_score(yva, p_attack)

threshold = 0.5
yhat = (p_attack >= threshold).astype(int)
print('\n=== Detecteur supervise (RandomForest) ===')
print(f'AUC detection: {auc_det:.4f}')
print('Confusion matrix [tn, fp; fn, tp]:')
print(confusion_matrix(yva, yhat))
print(classification_report(yva, yhat, digits=4))

imp = pd.DataFrame({
    'feature': feature_cols,
    'importance': clf.feature_importances_,
}).sort_values('importance', ascending=False)

print('Top indicateurs (importance):')
display(imp.head(10).round(4))

# 4) Detection non supervisee (cas reel: uniquement trafic benign pour apprentissage)
benign_mask = df_logs['label_attack'].values == 0
X_benign = X_feat[benign_mask]

iso = IsolationForest(
    n_estimators=300,
    contamination=0.10,
    random_state=SEED,
    n_jobs=-1,
 )
iso.fit(X_benign)

# anomaly_score: plus grand => plus suspect
raw_score = -iso.score_samples(X_feat)
auc_unsup = roc_auc_score(y_feat, raw_score)

print('\n=== Detecteur non supervise (IsolationForest) ===')
print(f'AUC (anomalie vs attaque): {auc_unsup:.4f}')

df_logs_eval = df_logs.copy()
df_logs_eval['anomaly_score'] = raw_score
display(
    df_logs_eval.groupby('label_attack')['anomaly_score']
    .describe()
    .round(4)
)

print('\nInterpretation rapide:')
print('- AUC proche de 0.5: patterns logs trop similaires, detection difficile.')
print('- AUC > 0.7: patterns detectables.')
print('- AUC > 0.85: signal fort pour detection operationnelle.')

,session_id,label_attack,n_queries,unique_ratio,repeat_ratio,p_mean,p_std,max_conf_mean,max_conf_std,entropy_mean,entropy_std,margin_mean,margin_std,near_boundary_ratio
0,benign_0,0,30,1.0,0.0,0.406805,0.474554,0.973451,0.098649,0.045958,0.167147,0.473451,0.098649,0.0
1,benign_1,0,30,1.0,0.0,0.281413,0.439862,0.984753,0.079206,0.026345,0.122860,0.484753,0.079206,0.0
2,benign_2,0,30,1.0,0.0,0.170991,0.371448,0.995672,0.023013,0.013236,0.068686,0.495672,0.023013,0.0
3,benign_3,0,30,1.0,0.0,0.200369,0.398344,0.998418,0.005984,0.007607,0.027899,0.498418,0.005984,0.0
4,benign_4,0,30,1.0,0.0,0.381293,0.477008,0.985124,0.079265,0.024055,0.123035,0.485124,0.079265,0.0


Sessions total: 480 | Benignes: 240 | Attaquantes: 240

=== Detecteur supervise (RandomForest) ===
AUC detection: 1.0000
Confusion matrix [tn, fp; fn, tp]:
[[72  0]
 [ 0 72]]
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000        72
           1     1.0000    1.0000    1.0000        72

    accuracy                         1.0000       144
   macro avg     1.0000    1.0000    1.0000       144
weighted avg     1.0000    1.0000    1.0000       144

Top indicateurs (importance):


,feature,importance
0,n_queries,0.5181
6,max_conf_std,0.0815
8,entropy_std,0.0808
10,margin_std,0.0799
7,entropy_mean,0.0546
5,max_conf_mean,0.0485
9,margin_mean,0.0453
4,p_std,0.0453
3,p_mean,0.0390
11,near_boundary_ratio,0.0071



=== Detecteur non supervise (IsolationForest) ===
AUC (anomalie vs attaque): 0.8184


,count,mean,std,min,25%,50%,75%,max
label_attack,,,,,,,,
0,240.0,0.4747,0.0485,0.4164,0.4414,0.4633,0.4912,0.7131
1,240.0,0.5517,0.0902,0.4496,0.4820,0.5099,0.6156,0.7834



Interpretation rapide:
- AUC proche de 0.5: patterns logs trop similaires, detection difficile.
- AUC > 0.7: patterns detectables.
- AUC > 0.85: signal fort pour detection operationnelle.
